# Corrective RAG (CRAG) — LangChain-Only Implementation

This notebook is a **single LangChain implementation** of the complete logic from `6_ambiguous.ipynb`.

### LangGraph → LangChain conversion
- `retrieve_node` → LangChain retriever `.invoke()`
- `eval_each_doc_node` → LangChain structured-output chain
- `rewrite_query_node` → LangChain structured-output chain
- `web_search_node` → LangChain Tavily tool
- `refine` → normal Python orchestration + LangChain filter chain
- `generate` → LangChain prompt + `ChatOpenAI`
- `route_after_eval` → normal Python `if/elif` routing
- `StateGraph`, `START`, `END`, `compile()` → removed completely

The original CRAG behavior is preserved, including the **CORRECT / AMBIGUOUS / INCORRECT** verdict logic and the corresponding knowledge sources.

In [1]:
# -----------------------------
# Imports + environment
# -----------------------------
from typing import List
from pydantic import BaseModel
import re
import os
from pprint import pp

from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from ddgs import DDGS

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

C:\Users\Uttam\AppData\Local\Temp\ipykernel_8004\1331709406.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 1. Load the knowledge base

Same three PDF sources as the original notebook.

In [2]:
# -----------------------------
# Data
# -----------------------------
docs = (
    PyPDFLoader("./documents/book1.pdf").load()
    + PyPDFLoader("./documents/book2.pdf").load()
    + PyPDFLoader("./documents/book3.pdf").load()
)

print(f"Loaded pages: {len(docs)}")

Loaded pages: 2123


In [3]:
# -----------------------------
# Chunking
# -----------------------------
chunks = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150,
).split_documents(docs)

# Preserve the original notebook's UTF-8 cleanup logic.
for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

print(f"Chunks created: {len(chunks)}")

Chunks created: 6399


In [4]:
# -----------------------------
# Vector store + retriever
# -----------------------------
embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",
    huggingfacehub_api_token=HF_TOKEN,
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="corrective_rag_demo"
)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

print("Vector store: Chroma")
print("Retriever: similarity, k=4")

Vector store: Chroma
Retriever: similarity, k=4


In [5]:
# -----------------------------
# LLM + thresholds
# -----------------------------
llm = ChatGroq(
                model="qwen/qwen3.8-27b",
                temperature=0,
                api_key=GROQ_API_KEY,
            )

UPPER_TH = 0.7
LOWER_TH = 0.3

## 2. Retrieval evaluator

Each retrieved document is independently scored from `0.0` to `1.0`.

- `score > 0.7` → evidence strong enough to classify retrieval as **CORRECT**
- all `score < 0.3` → **INCORRECT**
- otherwise → **AMBIGUOUS**

Documents with `score > 0.3` are retained as weakly relevant documents (`good_docs`).

In [6]:
# -----------------------------
# Structured output schema
# -----------------------------
class DocEvalScore(BaseModel):
    score: float
    reason: str


doc_eval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a strict retrieval evaluator for RAG.\n"
            "You will be given ONE retrieved chunk and a question.\n"
            "Return a relevance score in [0.0, 1.0].\n"
            "- 1.0: chunk alone is sufficient to answer fully/mostly\n"
            "- 0.0: chunk is irrelevant\n"
            "Be conservative with high scores.\n"
            "Also return a short reason.\n"
            "Output JSON only.",
        ),
        ("human", "Question: {question}\n\nChunk:\n{chunk}"),
    ]
)

doc_eval_chain = doc_eval_prompt | llm.with_structured_output(DocEvalScore, method="json_schema")


def evaluate_documents(question: str, retrieved_docs: List[Document]):
    """LangChain replacement for eval_each_doc_node."""
    scores: List[float] = []
    good_docs: List[Document] = []

    for doc in retrieved_docs:
        result = doc_eval_chain.invoke(
            {
                "question": question,
                "chunk": doc.page_content,
            }
        )

        scores.append(result.score)

        # Keep any doc above LOWER_TH as weakly relevant.
        if result.score > LOWER_TH:
            good_docs.append(doc)

    # CORRECT: at least one doc > UPPER_TH
    if any(score > UPPER_TH for score in scores):
        return {
            "good_docs": good_docs,
            "verdict": "CORRECT",
            "reason": f"At least one retrieved chunk scored > {UPPER_TH}.",
            "scores": scores,
        }

    # INCORRECT: all docs < LOWER_TH
    if len(scores) > 0 and all(score < LOWER_TH for score in scores):
        return {
            "good_docs": [],
            "verdict": "INCORRECT",
            "reason": f"All retrieved chunks scored < {LOWER_TH}.",
            "scores": scores,
        }

    # AMBIGUOUS: otherwise
    return {
        "good_docs": good_docs,
        "verdict": "AMBIGUOUS",
        "reason": f"No chunk scored > {UPPER_TH}, but not all were < {LOWER_TH}.",
        "scores": scores,
    }

## 3. Knowledge refinement

The original sentence-level decomposition and LLM filtering are preserved exactly.

Knowledge source by verdict:

| Verdict | Knowledge used for refinement |
|---|---|
| CORRECT | Internal `good_docs` only |
| AMBIGUOUS | Internal `good_docs` + web docs |
| INCORRECT | Web docs only |

In [7]:
# -----------------------------
# Sentence-level decomposer
# -----------------------------
def decompose_to_sentences(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if len(s.strip()) > 20]


# -----------------------------
# Sentence filter
# -----------------------------
class KeepOrDrop(BaseModel):
    keep: bool


filter_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a strict relevance filter.\n"
            "Return keep=true only if the sentence directly helps answer the question.\n"
            "Use ONLY the sentence. Output JSON only.",
        ),
        ("human", "Question: {question}\n\nSentence:\n{sentence}"),
    ]
)

filter_chain = filter_prompt | llm.with_structured_output(KeepOrDrop, method="json_schema")


def refine_knowledge(
    question: str,
    verdict: str,
    good_docs: List[Document],
    web_docs: List[Document],
):
    """LangChain/Python replacement for the original refine node."""

    if verdict == "CORRECT":
        docs_to_use = good_docs
    elif verdict == "INCORRECT":
        docs_to_use = web_docs
    else:  # AMBIGUOUS
        docs_to_use = good_docs + web_docs

    context = "\n\n".join(d.page_content for d in docs_to_use).strip()

    strips = decompose_to_sentences(context)
    print(f"\nNumber of Sentences Created Before Relevence Checking: {len(strips)}.")

    kept_strips: List[str] = []

    for sentence in strips:
        result = filter_chain.invoke(
            {
                "question": question,
                "sentence": sentence,
            }
        )

        if result.keep:
            kept_strips.append(sentence)
    print(f"\nNumber of Sentences Created After Relevence Checking: {len(kept_strips)}.")

    refined_context = "\n".join(kept_strips).strip()
    print(f"\nRefined Sentences Joined Together after Relevence Checking: {refined_context}.")

    return {
        "strips": strips,
        "kept_strips": kept_strips,
        "refined_context": refined_context,
    }

## 4. Knowledge searching

For `INCORRECT` and `AMBIGUOUS` retrieval, the original flow rewrites the question into a web-search query and searches Tavily with up to 5 results.

In [8]:
# -----------------------------
# Query rewrite
# -----------------------------
class WebQuery(BaseModel):
    query: str


rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Rewrite the user question into a web search query composed of keywords.\n"
            "Rules:\n"
            "- Keep it short (6–14 words).\n"
            "- If the question implies recency (e.g., recent/latest/last week/last month), add a constraint like (last 30 days).\n"
            "- Do NOT answer the question.\n"
            "- Return JSON with a single key: query",
        ),
        ("human", "Question: {question}"),
    ]
)

rewrite_chain = rewrite_prompt | llm.with_structured_output(WebQuery, method="json_schema")


def rewrite_query(question: str) -> str:
    result = rewrite_chain.invoke({"question": question})
    return result.query


# -----------------------------
# DuckDuckGo web search
# -----------------------------
ddgs = DDGS()


def web_search(web_query: str, fallback_question: str) -> List[Document]:

    q = web_query or fallback_question

    results = ddgs.text(
        q,
        max_results=1
    )

    web_docs: List[Document] = []

    for result in results or []:

        title = result.get("title", "")
        url = result.get("href", "")
        snippet = result.get("body", "")

        # Fetch actual webpage content
        try:
            extracted = ddgs.extract(url)
            content = extracted.get("content", "")
        except Exception:
            content = snippet

        text = (
            f"TITLE: {title}\n"
            f"URL: {url}\n"
            f"CONTENT:\n{content}"
        )

        web_docs.append(
            Document(
                page_content=text,
                metadata={
                    "url": url,
                    "title": title
                }
            )
        )

    return web_docs

## 5. Final generation

The generator uses only the refined context. The original question is passed separately.

In [9]:
# -----------------------------
# Generation chain
# -----------------------------
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful ML tutor. Answer ONLY using the provided context.\n"
            "If the context is empty or insufficient, say: 'I don't know + Reason that why you are saying I don't know in a same line.",
        ),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)

generation_chain = answer_prompt | llm


def generate_answer(question: str, refined_context: str) -> str:
    result = generation_chain.invoke(
        {
            "question": question,
            "context": refined_context,
        }
    )
    return result.content

## 6. Complete Corrective RAG pipeline — LangChain only

This function is the direct replacement for the LangGraph `StateGraph`.

The sequence is:

`Question → Retrieve → Evaluate → Route → (Web Search when needed) → Refine → Generate`

The routing is implemented with ordinary Python control flow; no LangGraph is used.

In [10]:
# -----------------------------
# Complete LangChain-only CRAG
# -----------------------------
def corrective_rag(question: str) -> dict:

    print("\n" + "=" * 80)
    print("RETRIEVED DOCUMENTS")
    print("=" * 80)

    # 1. Retrieve
    retrieved_docs = retriever.invoke(question)

    for i, doc in enumerate(retrieved_docs, start=1):

        print(f"\n{'─' * 80}")
        print(f"DOCUMENT {i}")
        print(f"{'─' * 80}")

        # Basic metadata
        print(f"ID       : {doc.id}")
        print(f"Title    : {doc.metadata.get('title', 'N/A')}")
        print(f"Source   : {doc.metadata.get('source', 'N/A')}")
        print(f"Page     : {doc.metadata.get('page_label', doc.metadata.get('page', 'N/A'))}")

        print(f"\nContent:")
        print(f"{doc.page_content.strip()[0:50]} ...")

    print("\n" + "=" * 80)

    "----------------------------------------------------------------------------------------------------------------------------------------------------------"

    print("\n" + "=" * 80)
    print("EVALUATE RETRIEVED KNOWLEDGE")
    print("=" * 80)

    # 2. Evaluate retrieved knowledge
    evaluation = evaluate_documents(question, retrieved_docs)
  
    verdict = evaluation["verdict"]
    reason = evaluation["reason"]
    good_docs = evaluation["good_docs"]
    scores = evaluation["scores"]

    print(f"Verdict: {verdict} \n")
    print(f"Reason: {reason} \n")
    print(f"Good Docs: {good_docs} \n")
    print(f"Number of good_docs: {len(good_docs)} \n")
    print(f"Scores: {scores}")

    # Match the original routing:
    # CORRECT -> refine directly
    # INCORRECT / AMBIGUOUS -> rewrite -> web search -> refine
    web_query = ""
    web_docs: List[Document] = []

    if verdict != "CORRECT":
        web_query = rewrite_query(question)
        print(f"\nWeb Query: {web_query}")
        web_docs = web_search(web_query, question)
        print(f"\nWeb Search Results: {web_docs}")

    print("\n" + "=" * 80)

    "----------------------------------------------------------------------------------------------------------------------------------------------------------"

    print("\n" + "=" * 80)
    print("KNOWLEDGE REFINEMENT")
    print("=" * 80)

    # 3. Knowledge refinement
    refinement = refine_knowledge(
        question=question,
        verdict=verdict,
        good_docs=good_docs,
        web_docs=web_docs,
    )

    print(f"Strips: {refinement["strips"]} \n")
    print(f"Kept Strips: {refinement["kept_strips"]} \n")
    print(f"Refined Context: {refinement["refined_context"]} \n")

    print("\n" + "=" * 80)

    "----------------------------------------------------------------------------------------------------------------------------------------------------------"

    print("\n" + "=" * 80)
    print("FINAL GENERATION")
    print("=" * 80)

    # 4. Final generation
    answer = generate_answer(
        question=question,
        refined_context=refinement["refined_context"],
    )

    print(f"Final Answer: {answer} \n")

    print("\n" + "=" * 80)
    
    # Return the complete state-like result without LangGraph.
    return {
        "question": question,
        "docs": retrieved_docs,
        "good_docs": good_docs,
        "verdict": verdict,
        "reason": reason,
        "scores": scores,
        "strips": refinement["strips"],
        "kept_strips": refinement["kept_strips"],
        "refined_context": refinement["refined_context"],
        "web_query": web_query,
        "web_docs": web_docs,
        "answer": answer,
    }

In [11]:
# -----------------------------
# Run example
# -----------------------------
res = corrective_rag(
    "What is Bias Variance Tradeoff?"
)

# print("VERDICT:", res["verdict"])
# print("REASON:", res["reason"])
# print("SCORES:", res["scores"])
# print("WEB_QUERY:", res["web_query"])
# print("WEB_DOCS:", len(res["web_docs"]))
# print("RETRIEVED_DOCS:", len(res["docs"]))
# print("GOOD_DOCS:", len(res["good_docs"]))
# print("KEPT_SENTENCES:", len(res["kept_strips"]))
# print("\nOUTPUT:\n", res["answer"])


RETRIEVED DOCUMENTS

────────────────────────────────────────────────────────────────────────────────
DOCUMENT 1
────────────────────────────────────────────────────────────────────────────────
ID       : 8785b937-3b93-41fe-acb7-cc28e33b9682
Title    : Pattern Recognition and Machine Learning
Source   : ./documents/book1.pdf
Page     : 149

Content:
(bias)2 =
∫
{ED[y(x; D)] − h(x)}2p(x)dx (3.42)
var ...


EVALUATE RETRIEVED KNOWLEDGE
Verdict: CORRECT 

Reason: At least one retrieved chunk scored > 0.7. 

Good Docs: [Document(id='8785b937-3b93-41fe-acb7-cc28e33b9682', metadata={'source': './documents/book1.pdf', 'total_pages': 758, 'moddate': '2008-02-08T16:41:33+01:00', 'producer': 'Acrobat Distiller 6.0 (Windows)', 'creator': 'Adobe Acrobat 6.0', 'page_label': '149', 'title': 'Pattern Recognition and Machine Learning', 'author': 'Christopher M. Bishop', 'page': 168, 'creationdate': '2006-10-18T12:52:36+08:00'}, page_content='(bias)2 =\n∫\n{ED[y(x; D)] − h(x)}2p(x)dx (3.42)\nvariance 

## 7. CRAG flow implemented by this notebook

```text
User Question
      │
      ▼
FAISS Similarity Retriever (k=4)
      │
      ▼
LLM Retrieval Evaluator
      │
      ├── score > 0.7 ───────────────► CORRECT
      │                                      │
      │                                      ▼
      │                              Internal good_docs
      │
      └── otherwise ─────────────────► Web Query Rewrite
                                             │
                                             ▼
                                        Tavily Search
                                             │
                         ┌───────────────────┴───────────────────┐
                         │                                       │
                     AMBIGUOUS                               INCORRECT
                         │                                       │
              good_docs + web_docs                           web_docs
                         │                                       │
                         └───────────────────┬───────────────────┘
                                             ▼
                                   Sentence Decomposition
                                             │
                                             ▼
                                   LLM Sentence Filter
                                             │
                                             ▼
                                      Refined Context
                                             │
                                             ▼
                                      Final LLM Answer
```

### Important
The original `6_ambiguous.ipynb` used LangGraph only for **state management and routing**. The actual retrieval, structured evaluation, query rewriting, Tavily search, sentence filtering, and generation can all be implemented with LangChain components plus ordinary Python orchestration. This notebook follows that approach.